In [5]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

dataset = pd.read_csv('CKD.csv')
dataset.dtypes

# Identify non-numeric (object) columns
categorical_cols = dataset.select_dtypes(include=['object']).columns
print("Categorical columns:", categorical_cols)

from sklearn.preprocessing import LabelEncoder

# Apply Label Encoding to each categorical column
le = LabelEncoder()
for col in categorical_cols:
    dataset[col] = le.fit_transform(dataset[col].astype(str))  # convert to string to avoid issues

# Verify preprocessing
print(dataset.head())
print(dataset.dtypes)

Categorical columns: Index(['sg', 'rbc', 'pc', 'pcc', 'ba', 'htn', 'dm', 'cad', 'appet', 'pe',
       'ane', 'classification'],
      dtype='object')
   age         bp  sg   al   su  rbc  pc  pcc  ba         bgr  ...        pcv  \
0  2.0  76.459948   2  3.0  0.0    1   0    0   0  148.112676  ...  38.868902   
1  3.0  76.459948   2  2.0  0.0    1   1    0   0  148.112676  ...  34.000000   
2  4.0  76.459948   0  1.0  0.0    1   1    0   0   99.000000  ...  34.000000   
3  5.0  76.459948   3  1.0  0.0    1   1    0   0  148.112676  ...  38.868902   
4  5.0  50.000000   2  0.0  0.0    1   1    0   0  148.112676  ...  36.000000   

             wc        rc  htn  dm  cad  appet  pe  ane  classification  
0   8408.191126  4.705597    0   0    0      1   1    0               1  
1  12300.000000  4.705597    0   0    0      1   0    0               1  
2   8408.191126  4.705597    0   0    0      1   0    0               1  
3   8408.191126  4.705597    0   0    0      1   0    1            

In [6]:
dataset["classification"].value_counts()

classification
1    249
0    150
Name: count, dtype: int64

In [7]:
dataset.columns

Index(['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu',
       'sc', 'sod', 'pot', 'hrmo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad',
       'appet', 'pe', 'ane', 'classification'],
      dtype='object')

In [8]:
indep=dataset[['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu',
       'sc', 'sod', 'pot', 'hrmo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad',
       'appet', 'pe', 'ane']]
dep=dataset["classification"]

In [10]:
#split into training set and test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(indep, dep, test_size = 1/3, random_state = 0)

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [11]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

param_grid = [
    {'solver': ['liblinear'], 'C': [0.1, 1, 10, 100, 1000], 'penalty':['l1','l2']},
    {'solver': ['lbfgs','newton-cg'], 'C': [0.1, 1, 10, 100, 1000], 'penalty':['l2']}
]

grid = GridSearchCV(LogisticRegression(), param_grid, refit = True, verbose = 3,n_jobs=-1, scoring='f1_weighted')

grid.fit(X_train, y_train)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


,estimator,LogisticRegression()
,param_grid,"[{'C': [0.1, 1, ...], 'penalty': ['l1', 'l2'], 'solver': ['liblinear']}, {'C': [0.1, 1, ...], 'penalty': ['l2'], 'solver': ['lbfgs', 'newton-cg']}]"
,scoring,'f1_weighted'
,n_jobs,-1
,refit,True
,cv,None
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,'l2'


In [17]:
re = grid.cv_results_

grid_prediction = grid.predict(X_test)

from sklearn.metrics import confusion_matrix, classification_report
cm = confusion_matrix(y_test, grid_prediction)
cm

clf_report = classification_report(y_test, grid_prediction)
print(clf_report)

              precision    recall  f1-score   support

           0       0.94      0.98      0.96        51
           1       0.99      0.96      0.98        82

    accuracy                           0.97       133
   macro avg       0.97      0.97      0.97       133
weighted avg       0.97      0.97      0.97       133



In [19]:
X_test[0]

array([-0.37073391, -0.50842064, -0.19971841, -0.68384276, -0.36409637,
        0.38924947,  0.51055216, -0.34299717, -0.23570226, -0.65023636,
       -0.24898008, -0.45519288,  0.42039161, -0.4834519 ,  1.45207747,
        0.49240365, -1.09016784,  2.23554046, -0.73926425, -0.73926425,
       -0.29277002,  0.51639778, -0.52223297, -0.44519456])